# AC 동손(AC Copper Loss) 비교 분석
## Morisco FEA-PEEC Hybrid vs. Ju Hybrid Analytical-FEA

### 참고 문헌
- **Morisco et al. (2020):** *Hybrid Analytical Model for AC Copper Loss Computation of Hairpin Winding* — FEA-PEEC 하이브리드, 로터 자기장 분리 포함
- **Ju / Volpe 방식:** *A Hybrid Analytical and FE-Based Method* — Dowell kR + FEA 슬롯 B-field + Popescu 중첩 원리

### 물리적 배경
$$\delta_\nu = \frac{1}{\sqrt{\pi \nu f_e \mu_0 \sigma}}, \quad \xi_\nu = \frac{h}{\delta_\nu}$$

$$k_R(m, \xi) = M(\xi) + \frac{(2m-1)^2}{3} Q(\xi)$$

$$P_{prox} = \frac{\sigma \omega^2 B^2 b h^3}{24} F_{prox}(\xi) \cdot L_a$$

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ACloss 폴더를 sys.path에 추가
MODULE_DIR = os.path.dirname(os.path.abspath("__file__"))
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

from morisco_acloss import (
    ConductorGeometry, SlotGeometry, OperatingPoint, RotorFieldInput,
    calculate_total_acloss_morisco, dc_loss_per_conductor, skin_depth, xi,
    dowell_kr
)
from ju_hybrid_acloss import (
    HairpinConductor, SlotLayout, WindingCurrentSpec, FEASlotField,
    calculate_acloss_ju, dowell_kr_at_layer, xi_nu, _dowell_M, _dowell_Q
)

print("모듈 로드 완료.")

## 1. 모터 파라미터 설정
**기준 모터:** 8극 48슬롯 Hairpin 권선 (전기차용 트랙션 모터)

In [ ]:
# ── 공통 물리 파라미터 ──────────────────────────────────────────────────────
B_COND    = 5.5e-3    # 도체 폭 [m]
H_COND    = 2.0e-3    # 도체 높이 [m]
SIGMA_CU  = 5.8e7     # 구리 도전율 @ 20°C [S/m]
L_ACTIVE  = 0.150     # 유효 스택 길이 [m]
W_SLOT    = 6.0e-3    # 슬롯 폭 [m]
N_LAYERS  = 6         # 슬롯당 도체 층수 (= turns/slot)
N_SLOTS_PH = 8        # 상당 슬롯 수 (48슬롯/3상/2층)

# Morisco 객체
m_cond = ConductorGeometry(b=B_COND, h=H_COND, sigma=SIGMA_CU, L_a=L_ACTIVE)
m_slot = SlotGeometry(w_slot=W_SLOT, n_L=N_LAYERS, n_par=1)

# Ju 객체
j_cond = HairpinConductor(b=B_COND, h=H_COND, sigma=SIGMA_CU, L_a=L_ACTIVE)
j_slot = SlotLayout(w_slot=W_SLOT, n_L=N_LAYERS, n_slot_phase=N_SLOTS_PH)

# ── 운전점 정의 ─────────────────────────────────────────────────────────────
OP1_F, OP1_I = 266.67,  240.0   # 4000 rpm, 240 Arms
OP2_F, OP2_I = 1066.67, 185.0   # 16000 rpm, 185 Arms

# 로터 자기장 (Morisco 전용)
rotor_input = RotorFieldInput(
    B_rotor_harmonics=(
        (1, 0.05),   # 기본 PM 조화파 0.05 T peak
        (3, 0.015),  # 3차 공간 조화파
        (5, 0.008),  # 5차 공간 조화파
    )
)

R_DC = L_ACTIVE / (SIGMA_CU * B_COND * H_COND)
print(f"도체 1개 DC 저항: {R_DC*1e6:.3f} μΩ")
print(f"슬롯 DC 손실 @ OP1: {R_DC * OP1_I**2 * N_LAYERS * 1e3:.2f} mW")

## 2. OP1 (기본속도: 4000 rpm) — 층별 손실 분석

In [ ]:
# Morisco 계산
m_op1 = OperatingPoint(f_e=OP1_F, I_rms=OP1_I)
res_m_op1 = calculate_total_acloss_morisco(m_cond, m_slot, m_op1, rotor_input, N_SLOTS_PH)

# Ju 계산
j_spec1 = WindingCurrentSpec.sinusoidal(f_e=OP1_F, I_rms=OP1_I)
res_j_op1 = calculate_acloss_ju(j_cond, j_slot, j_spec1)

layers = np.arange(1, N_LAYERS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"층별 AC 손실 — OP1: {OP1_F:.1f} Hz, {OP1_I} Arms", fontsize=13, fontweight='bold')

# 좌측: Morisco
ax = axes[0]
P_m_slot = res_m_op1['slot_result']['P_slot_per_conductor_W']
P_m_rotor_per = np.full(N_LAYERS, res_m_op1['P_rotor_total_W'] / N_LAYERS)
ax.bar(layers - 0.2, P_m_slot * 1e3, width=0.4, label='Slot field loss', color='tab:orange', alpha=0.85)
ax.bar(layers + 0.2, P_m_rotor_per * 1e3, width=0.4, label='Rotor field loss', color='tab:red', alpha=0.85)
ax.set_xlabel('도체 층 (1=슬롯 바닥)', fontsize=11)
ax.set_ylabel('AC 손실 [mW / 도체]', fontsize=11)
ax.set_title('Morisco 방법', fontsize=11)
ax.legend()
ax.grid(True, axis='y', ls='--', alpha=0.5)
ax.set_xticks(layers)

# 우측: Ju
ax = axes[1]
ax.bar(layers - 0.2, res_j_op1['P_skin_per_layer_W'] * 1e3, width=0.4,
       label='Skin effect', color='tab:blue', alpha=0.85)
ax.bar(layers + 0.2, res_j_op1['P_prox_per_layer_W'] * 1e3, width=0.4,
       label='Proximity effect', color='tab:green', alpha=0.85)
ax.set_xlabel('도체 층 (1=슬롯 바닥)', fontsize=11)
ax.set_ylabel('AC 손실 [mW / 도체]', fontsize=11)
ax.set_title('Ju 하이브리드 방법', fontsize=11)
ax.legend()
ax.grid(True, axis='y', ls='--', alpha=0.5)
ax.set_xticks(layers)

plt.tight_layout()
plt.show()

print(f"\n[Morisco] 슬롯 손실: {res_m_op1['P_slot_total_W']*1e3:.3f} mW, "
      f"로터 손실: {res_m_op1['P_rotor_total_W']*1e3:.3f} mW, "
      f"합계: {res_m_op1['P_total_per_slot_W']*1e3:.3f} mW")
print(f"[Ju]     스킨손실: {res_j_op1['P_skin_per_layer_W'].sum()*1e3:.3f} mW, "
      f"근접손실: {res_j_op1['P_prox_per_layer_W'].sum()*1e3:.3f} mW, "
      f"합계: {res_j_op1['P_total_slot_W']*1e3:.3f} mW")

## 3. OP1 vs OP2 — 두 운전점 비교

In [ ]:
# OP2 계산
m_op2 = OperatingPoint(f_e=OP2_F, I_rms=OP2_I)
res_m_op2 = calculate_total_acloss_morisco(m_cond, m_slot, m_op2, rotor_input, N_SLOTS_PH)

j_spec2 = WindingCurrentSpec.sinusoidal(f_e=OP2_F, I_rms=OP2_I)
res_j_op2 = calculate_acloss_ju(j_cond, j_slot, j_spec2)

# DC 손실 참조값
P_dc_op1 = R_DC * OP1_I**2 * N_LAYERS  # 슬롯당 DC 손실
P_dc_op2 = R_DC * OP2_I**2 * N_LAYERS

methods  = ['Morisco\n(OP1)', 'Ju\n(OP1)', 'Morisco\n(OP2)', 'Ju\n(OP2)']
p_total  = [
    res_m_op1['P_total_per_slot_W'],
    res_j_op1['P_total_slot_W'],
    res_m_op2['P_total_per_slot_W'],
    res_j_op2['P_total_slot_W'],
]
p_dc_ref = [P_dc_op1, P_dc_op1, P_dc_op2, P_dc_op2]
kR_eff   = [p / d for p, d in zip(p_total, p_dc_ref)]
colors   = ['tab:orange', 'tab:blue', 'tab:red', 'tab:green']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('OP1 vs OP2 — Morisco & Ju 방법 비교', fontsize=13, fontweight='bold')

bars = ax1.bar(methods, [p * 1e3 for p in p_total], color=colors, alpha=0.85, edgecolor='k', lw=0.7)
ax1.axhline(P_dc_op1 * 1e3, ls='--', color='gray', alpha=0.7, label='DC loss (OP1)')
ax1.axhline(P_dc_op2 * 1e3, ls=':', color='gray', alpha=0.7, label='DC loss (OP2)')
ax1.set_ylabel('슬롯당 AC 손실 [mW]', fontsize=11)
ax1.set_title('총 AC 손실 비교', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, axis='y', ls='--', alpha=0.5)
for bar, val in zip(bars, p_total):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val*1e3:.2f}', ha='center', va='bottom', fontsize=9)

bars2 = ax2.bar(methods, kR_eff, color=colors, alpha=0.85, edgecolor='k', lw=0.7)
ax2.axhline(1.0, ls='--', color='gray', alpha=0.7, label='kR = 1 (DC)')
ax2.set_ylabel('유효 kR = P_AC / P_DC', fontsize=11)
ax2.set_title('유효 AC 저항 계수 (kR)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, axis='y', ls='--', alpha=0.5)
for bar, val in zip(bars2, kR_eff):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 4. 주파수 스윕: kR vs Frequency

In [ ]:
freqs = np.logspace(1, 4, 250)   # 10 Hz ~ 10 kHz

kR_morisco = []
kR_ju_avg  = []

for f in freqs:
    # Morisco (슬롯 자기장만, 로터 미포함)
    op_sw = OperatingPoint(f_e=f, I_rms=OP1_I)
    res_sw = calculate_total_acloss_morisco(m_cond, m_slot, op_sw, rotor=None, n_slots_per_phase=1)
    P_dc_s = R_DC * OP1_I**2 * N_LAYERS
    kR_morisco.append(res_sw['P_total_per_slot_W'] / P_dc_s)

    # Ju (층 평균 kR)
    j_sw = WindingCurrentSpec.sinusoidal(f_e=f, I_rms=OP1_I)
    res_j_sw = calculate_acloss_ju(j_cond, j_slot, j_sw)
    kR_ju_avg.append(res_j_sw['P_total_slot_W'] / P_dc_s)

# Dowell 직접 계산 (각 층별)
kR_dowell_layers = {}
for m in range(1, N_LAYERS + 1):
    kR_dowell_layers[m] = [
        dowell_kr_at_layer(m, xi_nu(H_COND, f, 1, SIGMA_CU)) for f in freqs
    ]

fig, ax = plt.subplots(figsize=(11, 6))

# 층별 Dowell (음영)
kR_layer_arr = np.array([kR_dowell_layers[m] for m in range(1, N_LAYERS + 1)])
ax.fill_between(freqs, kR_layer_arr[0], kR_layer_arr[-1],
                alpha=0.15, color='gray', label='Dowell kR 범위 (L1~L6)')
for m in [1, 3, 6]:
    ax.semilogx(freqs, kR_dowell_layers[m], ls=':', lw=1.2,
                label=f'Dowell L{m} (m={m})')

ax.semilogx(freqs, kR_morisco, color='tab:red', lw=2.5, label='Morisco (슬롯 필드 평균)')
ax.semilogx(freqs, kR_ju_avg,  color='tab:blue', lw=2.5, ls='--', label='Ju 하이브리드 (층 평균)')

ax.axvline(OP1_F, ls='--', color='gray', alpha=0.7)
ax.text(OP1_F * 1.05, ax.get_ylim()[0] + 0.5, 'OP1', fontsize=9, color='gray')
ax.axvline(OP2_F, ls=':', color='gray', alpha=0.7)
ax.text(OP2_F * 1.05, ax.get_ylim()[0] + 0.5, 'OP2', fontsize=9, color='gray')

ax.set_xlabel('전기 주파수 [Hz]', fontsize=12)
ax.set_ylabel('유효 kR = P_AC / P_DC', fontsize=12)
ax.set_title('Morisco vs Ju — kR 주파수 응답 비교', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 5. PWM 고조파 영향 (Popescu 중첩 원리 — Ju 방법)

$$P_{ac} = \sum_{\nu} \left[ P_{skin,\nu} + P_{prox,\nu} \right], \quad P_{skin,\nu} = R_{dc} I_\nu^2 M(\xi_\nu)$$

In [ ]:
# 순수 정현파
spec_sine = WindingCurrentSpec.sinusoidal(f_e=OP1_F, I_rms=OP1_I)
res_sine  = calculate_acloss_ju(j_cond, j_slot, spec_sine)

# PWM 포함 (5, 7, 11, 13차 고조파)
spec_pwm = WindingCurrentSpec.with_pwm_harmonics(
    f_e=OP1_F, I1_rms=OP1_I,
    harmonic_pairs=[(5, 0.04), (7, 0.03), (11, 0.02), (13, 0.015)]
)
res_pwm = calculate_acloss_ju(j_cond, j_slot, spec_pwm)

print("=" * 60)
print("Ju 방법: 정현파 vs PWM 고조파 포함 비교 (OP1: 266.67 Hz)")
print("=" * 60)
print(f"정현파 기본파 손실: {res_sine['P_total_slot_W']*1e3:.4f} mW/slot")
print(f"PWM 포함 총 손실:   {res_pwm['P_total_slot_W']*1e3:.4f} mW/slot")
print(f"고조파 추가 손실:   {(res_pwm['P_total_slot_W']-res_sine['P_total_slot_W'])*1e3:.4f} mW")
print()
print("고조파별 손실 (PWM 케이스):")
for (nu, P_nu) in res_pwm['P_per_harmonic_W']:
    f_nu = OP1_F * nu
    delta_nu = 1.0 / np.sqrt(np.pi * f_nu * 4e-7*np.pi * SIGMA_CU)
    xi_val = H_COND / delta_nu
    print(f"  ν={nu:>2} ({f_nu:>8.2f} Hz): P={P_nu*1e3:>8.4f} mW,  δ={delta_nu*1e6:.1f} μm,  ξ={xi_val:.4f}")

# 시각화
nu_list = [h.order for h in spec_pwm.harmonics]
P_nu_list = [P for _, P in res_pwm['P_per_harmonic_W']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar([str(nu) for nu in nu_list], [p * 1e3 for p in P_nu_list],
              color=['tab:blue'] + ['tab:orange'] * (len(nu_list) - 1),
              alpha=0.85, edgecolor='k', lw=0.7)
ax.set_xlabel('전류 고조파 차수 ν', fontsize=12)
ax.set_ylabel('고조파별 슬롯 AC 손실 [mW]', fontsize=12)
ax.set_title('Ju 방법 — PWM 고조파별 AC 손실 (Popescu 중첩)', fontsize=12)
ax.grid(True, axis='y', ls='--', alpha=0.5)
for bar, val in zip(bars, P_nu_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val*1e3:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 6. 슬롯 B-field 시각화 (층별 자속 밀도 분포)

In [ ]:
from morisco_acloss import slot_leakage_B

layers_arr = np.arange(1, N_LAYERS + 1)
B_op1 = np.array([slot_leakage_B(k, N_LAYERS, W_SLOT, OP1_I) for k in layers_arr])
B_op2 = np.array([slot_leakage_B(k, N_LAYERS, W_SLOT, OP2_I) for k in layers_arr])

fig, ax = plt.subplots(figsize=(7, 5))
ax.step(np.append(B_op1 * 1e3, B_op1[-1] * 1e3),
        np.append(layers_arr, layers_arr[-1] + 1) - 0.5,
        where='post', color='tab:red', lw=2, label=f'OP1: {OP1_I} Arms')
ax.step(np.append(B_op2 * 1e3, B_op2[-1] * 1e3),
        np.append(layers_arr, layers_arr[-1] + 1) - 0.5,
        where='post', color='tab:blue', lw=2, ls='--', label=f'OP2: {OP2_I} Arms')
ax.scatter(B_op1 * 1e3, layers_arr, color='tab:red', zorder=5, s=50)
ax.scatter(B_op2 * 1e3, layers_arr, color='tab:blue', zorder=5, s=50)
ax.set_xlabel('슬롯 누설 자속 밀도 B [mT]', fontsize=12)
ax.set_ylabel('도체 층 번호 (1 = 슬롯 바닥)', fontsize=12)
ax.set_title('층별 슬롯 누설 자속 밀도 분포\n(Ampere 법칙 해석 모델)', fontsize=12)
ax.legend()
ax.grid(True, ls='--', alpha=0.5)
ax.set_yticks(layers_arr)
plt.tight_layout()
plt.show()

print("층별 슬롯 B [mT]:")
print(f"  OP1: " + ", ".join(f"L{k}={b:.2f}" for k, b in zip(layers_arr, B_op1 * 1e3)))
print(f"  OP2: " + ", ".join(f"L{k}={b:.2f}" for k, b in zip(layers_arr, B_op2 * 1e3)))

## 7. 종합 요약 표

In [ ]:
import pandas as pd

summary_data = []
for label, f_e, I_rms, res_m, res_j in [
    ('OP1 (4000rpm)', OP1_F, OP1_I, res_m_op1, res_j_op1),
    ('OP2 (16000rpm)', OP2_F, OP2_I, res_m_op2, res_j_op2),
]:
    P_dc_slot = R_DC * I_rms**2 * N_LAYERS
    delta = 1.0 / np.sqrt(np.pi * f_e * 4e-7*np.pi * SIGMA_CU)
    xi_val = H_COND / delta

    summary_data.append({
        '운전점': label,
        'f_e [Hz]': f_e,
        'I_rms [Arms]': I_rms,
        'δ [μm]': f'{delta*1e6:.1f}',
        'ξ = h/δ': f'{xi_val:.4f}',
        'P_DC [mW/slot]': f'{P_dc_slot*1e3:.3f}',
        'Morisco P_slot [mW]': f'{res_m["P_slot_total_W"]*1e3:.3f}',
        'Morisco P_rotor [mW]': f'{res_m["P_rotor_total_W"]*1e3:.3f}',
        'Morisco P_total [mW]': f'{res_m["P_total_per_slot_W"]*1e3:.3f}',
        'Morisco kR': f'{res_m["P_total_per_slot_W"]/P_dc_slot:.4f}',
        'Ju P_skin [mW]': f'{res_j["P_skin_per_layer_W"].sum()*1e3:.3f}',
        'Ju P_prox [mW]': f'{res_j["P_prox_per_layer_W"].sum()*1e3:.3f}',
        'Ju P_total [mW]': f'{res_j["P_total_slot_W"]*1e3:.3f}',
        'Ju kR': f'{res_j["P_total_slot_W"]/P_dc_slot:.4f}',
    })

df = pd.DataFrame(summary_data).set_index('운전점')
display(df.T)